In [2]:
# Step 1: Install scispacy and the model
#!pip install scispacy
#!pip install spacy
#!pip install https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/releases/v0.5.3/en_core_sci_sm-0.5.3.tar.gz

# Step 2: Import required libraries
import spacy
import scispacy
import pandas as pd
import math
import numpy as np
import requests
import re

# Step 3: Download and load the model
try:
    nlp = spacy.load("en_core_sci_sm")
    
    # Test the model
    text = "Paracetamol is commonly used to treat fever and mild to moderate pain."
    doc = nlp(text)
    
    # Print results
    print("Named Entities:")
    for ent in doc.ents:
        print(f"Entity: {ent.text}, Label: {ent.label_}")
        
except Exception as e:
    print(f"Error: {e}")
    print("\nTroubleshooting steps:")
    print("1. Make sure you're running this in a fresh Python session")
    print("2. Try restarting your Jupyter kernel")
    print("3. Verify your Python environment has write permissions")

Named Entities:
Entity: Paracetamol, Label: ENTITY
Entity: treat, Label: ENTITY
Entity: fever, Label: ENTITY
Entity: mild, Label: ENTITY
Entity: moderate pain, Label: ENTITY


/opt/anaconda3/envs/NLP/lib/python3.12/site-packages/spacy/language.py:2195: FutureWarning: Possible set union at position 6328
  deserializers["tokenizer"] = lambda p: self.tokenizer.from_disk(  # type: ignore[union-attr]


In [13]:
def get_rxcui(drug_name):
    """Get RxCUI from RxNorm for a given drug name."""
    url = f'https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name.lower()}'
    response = requests.get(url)
    data = response.json()
    return data.get('idGroup', {}).get('rxnormId', [None])[0]

def get_drug_MOA(rxcui):
    """Get drug classes for a given RxCUI using RxClass API."""
    if not rxcui:
        return []

    url = f'https://rxnav.nlm.nih.gov/REST/rxclass/class/byRxcui.json?rxcui={rxcui}'
    response = requests.get(url)
    data = response.json()

    classes = []
    for entry in data.get('rxclassDrugInfoList', {}).get('rxclassDrugInfo', []):
        class_name = entry.get('rxclassMinConceptItem', {}).get('className')
        class_type = entry.get('rxclassMinConceptItem', {}).get('classType')
        if class_name and class_type=="MOA":
            classes.append(f"{class_name} ({class_type})")
    return list(set(classes))  # Deduplicate

def classify_drug(drug_name):
    rxcui = get_rxcui(drug_name)
    if not rxcui:
        return f"{drug_name}: RxCUI not found"
    classes = get_drug_classes(rxcui)
    if not classes:
        return f"{drug_name}: No Mechanism of Action info found"
    return f"{drug_name}: " + "; ".join(classes)




In [6]:
# 🔍 Example usage
if __name__ == "__main__":
    drug_names = ["fluoxetine"]
    for drug in drug_names:
        print(classify_drug(drug))

fluoxetine: Serotonin Uptake Inhibitors (MOA); Monoamine Oxidase Inhibitors (MOA)


In [7]:
# Multiple ways to get and work with home directory
import os
from pathlib import Path

# 1. Using os.path
home_dir = os.path.expanduser('~')
print(f"Home directory (os.path): {home_dir}")

# 2. Using Path from pathlib (modern approach)
home = Path.home()
print(f"Home directory (pathlib): {home}")

# 3. Using environment variables
home_env = os.environ['HOME']  # For Unix/Linux/Mac
# home_env = os.environ['USERPROFILE']  # For Windows
print(f"Home directory (environ): {home_env}")

# Example: List contents of home directory
#print("\nContents of home directory:")
#for item in Path.home().iterdir():
#    print(f"- {item.name}")

# Create a new folder (if it doesn't exist)
new_folder = "/Users/naveendeevi/Datasets"
if not os.path.exists(new_folder):
    os.makedirs(new_folder)

# Change directory to the new folder
os.chdir(new_folder)

# Verify current working directory
print(f"Current working directory: {os.getcwd()}")

Home directory (os.path): /Users/naveendeevi
Home directory (pathlib): /Users/naveendeevi
Home directory (environ): /Users/naveendeevi
Current working directory: /Users/naveendeevi/Datasets


In [8]:
# Read the csv from the computer which has all the RXCUI and drug names.

df = pd.read_csv('Psych Drugs MOA.csv')

856796
[]
854834
['Monoamine Oxidase Inhibitors (MOA)']
1593855
['Monoamine Oxidase Inhibitors (MOA)']
856706
['Serotonin Uptake Inhibitors (MOA)', 'Dopamine Antagonists (MOA)', 'Adrenergic alpha-Antagonists (MOA)', 'Monoamine Oxidase Inhibitors (MOA)', 'Norepinephrine Uptake Inhibitors (MOA)']
856720
['Serotonin Uptake Inhibitors (MOA)', 'Dopamine Antagonists (MOA)', 'Adrenergic alpha-Antagonists (MOA)', 'Monoamine Oxidase Inhibitors (MOA)', 'Norepinephrine Uptake Inhibitors (MOA)']
856762
['Serotonin Uptake Inhibitors (MOA)', 'Norepinephrine Uptake Inhibitors (MOA)', 'Monoamine Oxidase Inhibitors (MOA)']
856769
['Serotonin Uptake Inhibitors (MOA)', 'GABA A Modulators (MOA)', 'Norepinephrine Uptake Inhibitors (MOA)', 'Monoamine Oxidase Inhibitors (MOA)']
856773
['Serotonin Uptake Inhibitors (MOA)', 'Norepinephrine Uptake Inhibitors (MOA)', 'Monoamine Oxidase Inhibitors (MOA)']
856783
['Serotonin Uptake Inhibitors (MOA)', 'Norepinephrine Uptake Inhibitors (MOA)', 'Monoamine Oxidase Inh

KeyboardInterrupt: 

In [30]:
def get_moa_from_rxnav(rxcui):
    """
    Get Mechanism of Action (MoA) for a drug using RxNav RxClass API with RxCUI.

    Parameters:
        rxcui (str): The RxNorm Concept Unique Identifier for the drug

    Returns:
        list[str]: MoA class names, or empty list if none found
    """
    base_url = f'https://rxnav.nlm.nih.gov/REST/rxclass/class/byRxcui.json?rxcui={rxcui}&relaSource=MEDRT&relas=has_MoA'

    try:
        response = requests.get(base_url)
        response.raise_for_status()
    except requests.exceptions.HTTPError as e:
        print(f"[Error] HTTP {response.status_code} for RxCUI '{rxcui}': {e}")
        return []
    except requests.exceptions.RequestException as e:
        print(f"[Error] Request failed: {e}")
        return []

    data = response.json()

    #print(data)
    
    classes = []
    for entry in data.get('rxclassDrugInfoList', {}).get('rxclassDrugInfo', []):
        class_name = entry.get('rxclassMinConceptItem', {}).get('className')
        class_type = entry.get('rxclassMinConceptItem', {}).get('classType')
        if class_name and class_type=="MOA":
            classes.append(f"{class_name} ({class_type})")
    return list(set(classes))  # Deduplicate

    """
    try:
        class_info = data["rxclassDrugInfoList"]["rxclassDrugInfo"]
        moa_classes = [entry["minConcept"]["name"] for entry in class_info]
        return moa_classes
    except KeyError:
        print(f"[Warning] No MoA found for RxCUI '{rxcui}'")
        return []
    """
# Test with fluoxetine's RxCUI: 4477
print(get_moa_from_rxnav(205535))


['fluoxetine']


In [25]:
rx = 205535
base_url = f'https://rxnav.nlm.nih.gov/REST/rxclass/class/byRxcui.json?rxcui={rx}&relaSource=MEDRT&relas=has_MoA'

#params = {"relaSource": "MEDRT","relas": "has_MoA"}
response = requests.get(base_url)
print(response.json())

{'rxclassDrugInfoList': {'rxclassDrugInfo': [{'minConcept': {'rxcui': '4493', 'name': 'fluoxetine', 'tty': 'IN'}, 'rxclassMinConceptItem': {'classId': 'N0000000109', 'className': 'Serotonin Uptake Inhibitors', 'classType': 'MOA'}, 'rela': 'has_moa', 'relaSource': 'MEDRT'}]}}


In [31]:
### Class name from RXCUI

class_x = []
for i in df['Rxcui']:
    print(i)
    if isinstance(i, float) and math.isnan(i):
        value = ""
    elif str(i).isdigit():
        value = get_moa_from_rxnav(i)
    else: 
        value = ""
    print(value)
    if len(value) == 0: 
        x = "" 
    else: 
        x = value

856796
[Warning] No MoA found for RxCUI '856796'
[]
854834
[Warning] No MoA found for RxCUI '854834'
[]
1593855
[Warning] No MoA found for RxCUI '1593855'
[]
856706
['amitriptyline hydrochloride', 'amitriptyline hydrochloride', 'amitriptyline', 'amitriptyline', 'perphenazine', 'perphenazine']
856720
['amitriptyline hydrochloride', 'amitriptyline hydrochloride', 'amitriptyline', 'amitriptyline', 'perphenazine', 'perphenazine']
856762
['amitriptyline hydrochloride', 'amitriptyline hydrochloride', 'amitriptyline', 'amitriptyline']
856769
['amitriptyline hydrochloride', 'amitriptyline hydrochloride', 'chlordiazepoxide', 'amitriptyline', 'amitriptyline']
856773
['amitriptyline hydrochloride', 'amitriptyline hydrochloride', 'amitriptyline', 'amitriptyline']
856783
['amitriptyline hydrochloride', 'amitriptyline hydrochloride', 'amitriptyline', 'amitriptyline']
856792
['amitriptyline hydrochloride', 'amitriptyline hydrochloride', 'chlordiazepoxide', 'amitriptyline', 'amitriptyline']
856797
['a

KeyboardInterrupt: 

In [49]:
def get_chembl_moa(drug_name):
    # Step 1: Search for the ChEMBL ID
    search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    res = requests.get(search_url).json()
    
    try:
        chembl_id = res['molecules'][0]['molecule_chembl_id']
    except (KeyError, IndexError):
        print(f"Drug '{drug_name}' not found.")
        return []

    # Step 2: Get MoA
    moa_url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"
    moa_data = requests.get(moa_url).json()

    #print(moa_data)
    
    try:
        return [
            {
                "mechanism": m["mechanism_of_action"],
                "target": m["target_chembl_id"],
                "target_name": m.get("target_name"),
                "action_type": m["action_type"]
            }
            for m in moa_data["mechanisms"]
        ]
    except KeyError:
        return []

# Example usage
print(get_chembl_moa("Biperiden"))

[{'mechanism': 'Muscarinic acetylcholine receptor M1 antagonist', 'target': 'CHEMBL216', 'target_name': None, 'action_type': 'ANTAGONIST'}]


In [45]:
import requests

def get_primary_targets_from_chembl(drug_name):
    """
    Get the primary target(s) of a drug using ChEMBL API.
    
    Parameters:
        drug_name (str): Name of the drug (e.g., "gabapentin")
    
    Returns:
        list[dict]: List of primary targets with mechanism and target info
    """
    # Step 1: Search for the ChEMBL ID of the drug
    search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    res = requests.get(search_url).json()
    
    try:
        chembl_id = res['molecules'][0]['molecule_chembl_id']
    except (KeyError, IndexError):
        print(f"[Error] Drug '{drug_name}' not found in ChEMBL.")
        return []

    # Step 2: Get mechanisms and filter for PRIMARY action_type
    moa_url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"
    moa_data = requests.get(moa_url).json()

    try:
        primary_targets = [
            {
                "mechanism": m["mechanism_of_action"],
                "target_name": m["target_pref_name"],
                "target_chembl_id": m["target_chembl_id"]
            }
            for m in moa_data["mechanisms"]
            if m.get("action_type") == "PRIMARY"
        ]
        return primary_targets
    except KeyError:
        print(f"[Warning] No mechanism data found for '{drug_name}'.")
        return []

# 🔬 Example usage:
print(get_primary_targets_from_chembl("Biperiden"))


[]


In [50]:
def get_chembl_moa(drug_name):
    # Step 1: Search for the ChEMBL ID
    search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    res = requests.get(search_url).json()
    
    try:
        chembl_id = res['molecules'][0]['molecule_chembl_id']
    except (KeyError, IndexError):
        print(f"Drug '{drug_name}' not found.")
        return []

    # Step 2: Get MoA
    moa_url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"
    moa_data = requests.get(moa_url).json()
    mechanisms = moa_data.get("mechanisms", [])

    results = []

    for m in mechanisms:
        target_id = m.get("target_chembl_id")
        target_name = None

        # Fetch target name using target_chembl_id if available
        if target_id:
            target_url = f"https://www.ebi.ac.uk/chembl/api/data/target/{target_id}.json"
            target_res = requests.get(target_url).json()
            target_name = target_res.get("pref_name")

        results.append({
            "mechanism": m.get("mechanism_of_action"),
            "target_chembl_id": target_id,
            "target_name": target_name,
            "action_type": m.get("action_type")
        })

    return results

# Example usage
print(get_chembl_moa("Biperiden"))

[{'mechanism': 'Muscarinic acetylcholine receptor M1 antagonist', 'target_chembl_id': 'CHEMBL216', 'target_name': 'Muscarinic acetylcholine receptor M1', 'action_type': 'ANTAGONIST'}]


In [51]:
rst = get_chembl_moa("Biperiden")

In [54]:
import requests

def has_generic_version(drug_name):
    # Step 1: Get RxCUI for the drug
    rxcui_url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name}"
    try:
        response = requests.get(rxcui_url)
        response.raise_for_status()
        res = response.json()
    except Exception as e:
        print(f"[Error] Failed to fetch or parse RxCUI: {e}")
        return []

    rxcui = res.get("idGroup", {}).get("rxnormId", [None])[0]
    if not rxcui:
        print(f"[Info] No RxCUI found for '{drug_name}'")
        return []

    # Step 2: Fetch related concepts (looking for generic types)
    related_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/allrelated.json"
    try:
        rel_response = requests.get(related_url)
        rel_response.raise_for_status()
        related_data = rel_response.json()
    except Exception as e:
        print(f"[Error] Failed to fetch related concepts: {e}")
        return []

    # Step 3: Extract generic-like concepts
    concept_groups = related_data.get("allRelatedGroup", {}).get("conceptGroup", [])
    generics = []

    for group in concept_groups:
        tty = group.get("tty")
        if tty in ["SCD", "SCDC", "GPCK"]:  # likely generic types
            for concept in group.get("conceptProperties", []):
                generics.append(concept.get("name"))

    return generics

# Example usage
print(has_generic_version("Atomoxetine"))



['atomoxetine 10 MG Oral Capsule', 'atomoxetine 18 MG Oral Capsule', 'atomoxetine 25 MG Oral Capsule', 'atomoxetine 40 MG Oral Capsule', 'atomoxetine 60 MG Oral Capsule', 'atomoxetine 100 MG Oral Capsule', 'atomoxetine 80 MG Oral Capsule', 'atomoxetine 10 MG', 'atomoxetine 18 MG', 'atomoxetine 25 MG', 'atomoxetine 40 MG', 'atomoxetine 60 MG', 'atomoxetine 100 MG', 'atomoxetine 80 MG']


In [57]:
import requests
import pandas as pd

def get_generic_versions_from_df(df):
    """
    Check if drugs in a Pandas DataFrame have generic formulations using RxNorm.

    Parameters:
        df (pd.DataFrame): DataFrame with columns 'rxcui', 'medication_name', 'medication_first_name'

    Returns:
        pd.Series: A Series with generic versions (if any), or empty list
    """
    def get_generic_versions(row):
        # Extract RxCUI or fallback to medication_name or medication_first_name
        rxcui = row.get("rxcui")
        drug_name = row.get("medication_name") or row.get("medication_first_name")

        # Step 1: Get RxCUI if not provided
        if not rxcui and drug_name:
            try:
                url = f"https://rxnav.nlm.nih.gov/REST/rxcui.json?name={drug_name}"
                res = requests.get(url)
                res.raise_for_status()
                data = res.json()
                rxcui = data.get("idGroup", {}).get("rxnormId", [None])[0]
            except Exception as e:
                print(f"[Error] RxCUI lookup failed for '{drug_name}': {e}")
                return []

        if not rxcui:
            print("[Info] No RxCUI or drug name available.")
            return []

        # Step 2: Get all related concepts
        try:
            related_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/allrelated.json"
            rel_res = requests.get(related_url)
            rel_res.raise_for_status()
            related_data = rel_res.json()
        except Exception as e:
            print(f"[Error] Failed to fetch related concepts for RxCUI {rxcui}: {e}")
            return []

        # Step 3: Extract likely generic drug concepts
        concept_groups = related_data.get("allRelatedGroup", {}).get("conceptGroup", [])
        generic_names = []

        for group in concept_groups:
            if group.get("tty") in ["SCD", "SCDC", "GPCK"]:
                for concept in group.get("conceptProperties", []):
                    name = concept.get("name")
                    if name:
                        generic_names.append(name)

        return generic_names

    # Apply function across DataFrame rows
    return df.apply(get_generic_versions, axis=1)

# Example usage
#df = pd.DataFrame({
#    'rxcui': ['197361', '5361'],  # Example RxCUI values
#    'medication_name': ['Atomoxetine', 'Gabapentin'],
#    'medication_first_name': ['Lisinopril', '']
#})

#df['generic_versions'] = get_generic_versions_from_df(df)
#print(df)


In [58]:
import requests

def get_side_effects_from_fda(drug_name):
    url = f"https://api.fda.gov/drug/label.json?search=openfda.brand_name:\"{drug_name}\""
    response = requests.get(url)
    data = response.json()
    
    side_effects = []
    try:
        # Extract side effects from the returned data
        for result in data.get("results", []):
            if 'adverse_reactions' in result:
                side_effects.extend(result["adverse_reactions"])
    except KeyError:
        print("No side effects information found for this drug.")
    
    return side_effects

# Example Usage
side_effects = get_side_effects_from_fda("Alprazolam")
print(side_effects)


['6 ADVERSE REACTIONS The following clinically significant adverse reactions are described elsewhere in the labeling: Risks from Concomitant Use with Opioids [see Warnings and Precautions (5.1) ] Abuse, Misuse, and Addiction [see Warnings and Precautions (5.2) ] Dependence and Withdrawal Reactions [see Warnings and Precautions (5.3) ] Effects on Driving and Operating Machinery [see Warnings and Precautions (5.4) ] Patients with Depression [see Warnings and Precautions (5.6) ] Neonatal Sedation and Withdrawal Syndrome [see Warnings and Precautions (5.8) ] Risks in Patients with Impaired Respiratory Function [see Warnings and Precautions (5.9) ] The most common adverse reactions reported in clinical trials for generalized anxiety disorder and panic disorder (incidence > 5% and at least twice that of placebo) include: impaired coordination, hypotension, dysarthria, and increased libido. ( 6.1 ) To report SUSPECTED ADVERSE REACTIONS, contact Aurobindo Pharma USA, Inc. at 1-866-850-2876 or 

In [59]:
def get_chembl_moa(drug_name):
    # Step 1: Search for the ChEMBL ID
    search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    res = requests.get(search_url).json()
    
    try:
        chembl_id = res['molecules'][0]['molecule_chembl_id']
    except (KeyError, IndexError):
        print(f"Drug '{drug_name}' not found.")
        return []

    # Step 2: Get MoA
    moa_url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"
    moa_data = requests.get(moa_url).json()
    mechanisms = moa_data.get("mechanisms", [])

    results = []

    for m in mechanisms:
        target_id = m.get("target_chembl_id")
        target_name = None

        # Fetch target name using target_chembl_id if available
        if target_id:
            target_url = f"https://www.ebi.ac.uk/chembl/api/data/target/{target_id}.json"
            target_res = requests.get(target_url).json()
            target_name = target_res.get("pref_name")

        results.append({
            "mechanism": m.get("mechanism_of_action"),
            "target_chembl_id": target_id,
            "target_name": target_name,
            "action_type": m.get("action_type")
        })

    return results

# Example usage
print(get_chembl_moa("Biperiden"))

[{'mechanism': 'Muscarinic acetylcholine receptor M1 antagonist', 'target_chembl_id': 'CHEMBL216', 'target_name': 'Muscarinic acetylcholine receptor M1', 'action_type': 'ANTAGONIST'}]


In [3]:
def get_side_effects_tables(drug_name):
    # Step 1: Call OpenFDA API
    url = f"https://api.fda.gov/drug/label.json?search=openfda.brand_name:\"{drug_name}\"&limit=1"
    response = requests.get(url)

    if response.status_code != 200:
        print("Failed to fetch data.")
        return []

    data = response.json()

    # Step 2: Extract adverse_reactions_table (if exists)
    try:
        html_tables = data['results'][0]['adverse_reactions_table']
    except (KeyError, IndexError):
        print("No adverse reactions table found.")
        return []

    # Step 3: Convert HTML to DataFrames
    tables = []
    for html in html_tables:
        try:
            df_list = pd.read_html(html)  # requires lxml
            tables.extend(df_list)
        except Exception as e:
            print(f"Failed to parse table: {e}")
    
    return tables

# 🔍 Example usage
drug = "Alprazolam"
tables = get_side_effects_tables(drug)

# Display the extracted tables
for i, table in enumerate(tables):
    print(f"\n--- Table {i+1} ---")
    print(table)


--- Table 1 ---
                                                   0                     1  \
0                                                NaN      Alprazolam n=565   
1  Nervous system disorders Drowsiness Light-head...  41% 21% 2% 2% 15% 4%   
2  Cardiovascular disorders Hypotension Skin and ...                 5% 4%   

                      2  
0         Placebo n=505  
1  22% 19% 1% 1% 13% 2%  
2                 2% 3%  

--- Table 2 ---
                                                   0  \
0                                                NaN   
1  Drowsiness Fatique and Tiredness Impaired Coor...   
2  Gastrointestinal disorders Constipation Increa...   
3        Skin and subcutaneous tissue disorders Rash   
4  Other Increased appetite Decreased appetite We...   

                                                   1  \
0                                  Alprazolam n=1388   
1  77% 49% 40% 33% 33% 29% 14% 23% 10% 8% 7% 3% 2...   
2                                           

/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_30923/1441899102.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_list = pd.read_html(html)  # requires lxml
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_30923/1441899102.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_list = pd.read_html(html)  # requires lxml
/var/folders/4n/y26qxrlx1rg5fr42sym5pf9c0000gn/T/ipykernel_30923/1441899102.py:23: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df_list = pd.read_html(html)  # requires lxml


In [4]:
table

,0,1
0,NaN,Alprazolam-treated Patients n=641
1,Nervous system disorders Insomnia Light-headed...,29.5% 19.3% 17.3% 17.0% 6.9% 6.6% 5.9% 5.8%
2,Psychiatric disorders Anxiety Fatigue and Tire...,19.2% 18.4% 10.5% 10.3% 5.5% 5.1% 5.0%
3,Gastrointestinal disorders Nausea/Vomiting Dia...,16.5% 13.6% 10.6%
4,Metabolism and nutrition disorders Weight loss...,13.3% 12.8%
5,Dermatological disorders Sweating,14.4%
6,Cardiovascular disorders Tachycardia,12.2%
7,Special Senses Blurred vision,10.0%
8,n=number of patients.,n=number of patients.


In [5]:
def get_chembl_moa(drug_name):
    # Step 1: Search for the ChEMBL ID
    search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
    res = requests.get(search_url).json()
    
    try:
        chembl_id = res['molecules'][0]['molecule_chembl_id']
    except (KeyError, IndexError):
        print(f"Drug '{drug_name}' not found.")
        return []

    # Step 2: Get MoA
    moa_url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"
    moa_data = requests.get(moa_url).json()
    mechanisms = moa_data.get("mechanisms", [])

    results = []

    for m in mechanisms:
        target_id = m.get("target_chembl_id")
        target_name = None

        # Fetch target name using target_chembl_id if available
        if target_id:
            target_url = f"https://www.ebi.ac.uk/chembl/api/data/target/{target_id}.json"
            target_res = requests.get(target_url).json()
            target_name = target_res.get("pref_name")

        results.append({
            "mechanism": m.get("mechanism_of_action"),
            "target_chembl_id": target_id,
            "target_name": target_name,
            "action_type": m.get("action_type")
        })

    return results


In [8]:
get_chembl_moa('Amphetamine/dextroamphetamine 5 mg cp24')

[]

In [9]:
def get_chembl_moa_from_rxcui(rxcui):
    # Step 1: Get drug name from RxCUI
    try:
        rxnav_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/properties.json"
        rxnav_res = requests.get(rxnav_url)
        rxnav_res.raise_for_status()
        drug_name = rxnav_res.json().get("properties", {}).get("name")
    except Exception as e:
        print(f"[Error] Could not resolve drug name from RxCUI {rxcui}: {e}")
        return []

    if not drug_name:
        print(f"[Error] No drug name found for RxCUI {rxcui}")
        return []

    # Step 2: Search for the ChEMBL ID
    try:
        search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
        res = requests.get(search_url).json()
        chembl_id = res['molecules'][0]['molecule_chembl_id']
    except (KeyError, IndexError, Exception) as e:
        print(f"Drug '{drug_name}' not found in ChEMBL: {e}")
        return []

    # Step 3: Get MoA
    moa_url = f"https://www.ebi.ac.uk/chembl/api/data/mechanism.json?molecule_chembl_id={chembl_id}"
    moa_data = requests.get(moa_url).json()
    mechanisms = moa_data.get("mechanisms", [])

    results = []

    for m in mechanisms:
        target_id = m.get("target_chembl_id")
        target_name = None

        # Fetch target name using target_chembl_id if available
        if target_id:
            target_url = f"https://www.ebi.ac.uk/chembl/api/data/target/{target_id}.json"
            target_res = requests.get(target_url).json()
            target_name = target_res.get("pref_name")

        results.append({
            "mechanism": m.get("mechanism_of_action"),
            "target_chembl_id": target_id,
            "target_name": target_name,
            "action_type": m.get("action_type")
        })

    return results

In [11]:
get_chembl_moa_from_rxcui("221057")

[{'mechanism': 'Dopamine transporter releasing agent',
  'target_chembl_id': 'CHEMBL238',
  'target_name': 'Dopamine transporter',
  'action_type': 'RELEASING AGENT'},
 {'mechanism': 'Norepinephrine transporter releasing agent',
  'target_chembl_id': 'CHEMBL222',
  'target_name': 'Norepinephrine transporter',
  'action_type': 'RELEASING AGENT'}]

In [12]:
def get_chembl_indications_improved_test(drug_list):
    unique_ids = set()
    for drug_name in drug_list:
        try:
            search_url = f"https://www.ebi.ac.uk/chembl/api/data/molecule/search.json?q={drug_name}"
            res = requests.get(search_url).json()
            chembl_ids = [m["molecule_chembl_id"] for m in res.get("molecules", []) if "molecule_chembl_id" in m]
            unique_ids.update(chembl_ids)
        except Exception as e:
            print(f"[Warning] Could not fetch ChEMBL ID for '{drug_name}': {e}")
            continue
    return unique_ids

In [13]:
test = ['diphenhydramine hydrochloride 50 MG/ML','haloperidol 5 MG/ML','hydroxyzine pamoate']

In [15]:
test_2_set = get_chembl_indications_improved_test(test)

In [34]:
test_3_set = {'CHEMBL2219425','CHEMBL1112','CHEMBL2219424','CHEMBL4205525'}

In [37]:
def get_chembl_indications_improved_indication_dict(set_ip):
    chembl_conditions = {}
    for i in set_ip:
        print(i)
        ind_url = f"https://www.ebi.ac.uk/chembl/api/data/drug_indication.json?molecule_chembl_id={i}"
        print(ind_url)
        ind_data = requests.get(ind_url).json()
        print(ind_data)
        try:
            for item in ind_data.get("drug_indications", []):
                #indications.append(item.get("efo_term"))
                chembl_conditions[item] = item.get("efo_term")
        except (KeyError, IndexError):
            continue
    
    #indications = list(dict.fromkeys(indications))
    #print(indications)
    #if len(indications) > 0: 
    #    return "; ".join(map(str, indications))
    #else: 
    #    return None
    return chembl_conditions


In [50]:
x = get_chembl_indications_improved_indication_dict(test_3_set)

CHEMBL2219425


KeyboardInterrupt: 

In [51]:
ind_url = f"https://www.ebi.ac.uk/chembl/api/data/drug_indication.json?molecule_chembl_id=CHEMBL2219425"
print(ind_url)
ind_data = requests.get(ind_url)
print(ind_data)
# Check if request was successful
if ind_data.status_code == 200:
    #json_data = response.json()
    json_text = ind_data.text
    #info = json.loads(json_text)
    print(json_text)
    #print(json_text['drug_indications'])
#print(info)

https://www.ebi.ac.uk/chembl/api/data/drug_indication.json?molecule_chembl_id=CHEMBL2219425
<Response [500]>
